### Clean Data

In [ ]:
import pandas as pd
from src.config import *
import numpy as np
import re

In [ ]:
data=pd.read_csv(DIR_RAW_DATA, low_memory=False)
survey = pd.read_excel(DIR_META_DATA, sheet_name='survey')
choices = pd.read_excel(DIR_META_DATA, sheet_name='choices')

In [ ]:
survey=survey.loc[:,["type","name","label","hint", "required"]]

### Convert Data Types

In [ ]:
# Convert Duration to Minutes
data['duration'] = data['duration'] / 60

### Exclude Submissions

#### Exclude based on Missingness

In [ ]:
#All columsn with 100% missingness are removed.
cols_to_drop=data.loc[:,data.isna().mean()==1].columns.tolist()
print(f"Removing {len(cols_to_drop)} columns with 100% missingness")
data = data.loc[:,~data.columns.isin(cols_to_drop)]
survey=survey[~survey.name.isin(cols_to_drop)]

#### Exclude unfinished submissions

In [ ]:

DIR_VALIDATE = Path("data/4_validation_checks/")

In [ ]:
# Exclude submissions with more than 90% missingness, ie. unfinsihed submissions
rows_to_drop=data.index[data.isna().sum(axis=1)/data.shape[1]>0.9].tolist()
print(f"Removing {len(rows_to_drop)} submissions with more than 90% missingness")    
data.iloc[rows_to_drop].to_csv(DIR_VALIDATE/"00_unfinished_submissions.csv",index=False, encoding='utf-8-sig')
data=data.drop(index=rows_to_drop)

#### Exclude based on Location

In [ ]:
# outsiders = data.loc[data.location == -99]
# outsiders_idx = outsiders.index.tolist()

# data = data.drop(outsiders_idx)

# print(f"Removing {len(outsiders)} submissions of location {outsiders.location.unique()}.")

#### Exclude based on Duration

In [ ]:
dur_data=data.copy()
DURATION_THRESHOLD=18
rows_to_drop=data[data.duration<DURATION_THRESHOLD].index.tolist()
print(f"Removing {len(rows_to_drop)} submissions with duration <={DURATION_THRESHOLD}")
data.iloc[rows_to_drop].to_csv(DIR_VALIDATE/"00_short_duration.csv",index=False, encoding='utf-8-sig') 
data=data.drop(index=rows_to_drop)


#### Exploring Duration Distribution

In [ ]:
dur_data.duration.describe()

In [ ]:
missing_rowwise=dur_data.isna().mean(axis=1).mean()
missing_colwise=dur_data.isna().mean()
print(f"{round(missing_rowwise*100,2)}% of submission fields per submission is empty.")

In [ ]:
short_duration_indexes = dur_data[dur_data.duration <= DURATION_THRESHOLD].index.tolist()

In [ ]:
short_duration_data=dur_data.loc[short_duration_indexes]
sd_missing_rowwise=short_duration_data.isna().mean(axis=1).mean()
sd_missing_colwise=short_duration_data.isna().mean()
print(f"{round(sd_missing_rowwise*100,2)}% of submission fields per submission is empty.")

### Group Text Data
prioritize: 
- station_most_listened_other
- station_past_7_days_other
- station_listened_yesterday_other
- radio_what_other
- method_other: all NAs


Free-text "other" radio stations are standardised by `src/stations.py` (replaces the `radio_matching/` lookups): each answer becomes one entry per station, via, in order, the reviewed lookup CSV (`DIR_STATIONS_OTHER_CLEANED`), `STATION_PATTERNS` rules, fuzzy matching against known names and the questionnaire's choice labels, and otherwise kept as typed and flagged. The FM frequency written next to a station is kept; a mention without one takes the nearest respondent's (same station, within `STATIONS_MAX_KM`). Review what it does with `python 01b_review_stations.py`.

#### Clean station_*_other columns and merge into station_* columns

In [ ]:
from src import stations

# Put each answer's standardised stations in place of -22 in the select_multiple column:
#  - a station matching a questionnaire choice becomes that choice's code (counted once if also ticked)
#  - any other station becomes one token with spaces as underscores ("Radio_Nouvelle_Etoile_94.4");
#    01_mapping keeps unknown tokens as-is and turns "_" back into spaces after mapping
#  - -88/-99 on their own are kept; mixed with stations they're dropped
def normalize_88_col(s):
    return s.astype("string").str.replace(r"^\s*88\s*$", "-88", regex=True)

cols = ["station_most_listened", "station_listened_yesterday", "station_past_7_days"]
station_labels = {}
for col in cols:
    station_labels[col], qtype = stations.station_choices(DIR_META_DATA, col)
    assert qtype == "select_multiple", f"{col} is {qtype}; merge_other expects select_multiple"

station_lookup = stations.load_lookup(DIR_STATIONS_OTHER_CLEANED)
std = stations.StationStandardiser(
    lookup=station_lookup,
    extra_names=[label for labels in station_labels.values() for label in labels])
data, station_mentions = stations.combine_station_columns(data, cols, std, station_labels, max_km=STATIONS_MAX_KM)
for col in cols:
    data[col] = normalize_88_col(data[col])

print("Station 'other' mentions by method:")
print(station_mentions["method"].value_counts().to_string())

# Review files (no respondent rows) in DIR_VALIDATE: which write-ins matched a questionnaire choice,
# which stations are not on the questionnaire, and a draft reviewed lookup to correct and save as
# DIR_STATIONS_OTHER_CLEANED.
stations.export_review(station_mentions, station_labels, DIR_VALIDATE, lookup=station_lookup)

In [ ]:
# The *_other columns are kept (raw answers) so 01b_review_stations.py can review the standardisation;
# their stations are already merged into the select_multiple columns above.

### Create New Columns


#### use

- user: currently using
- future user: currently not using, but considering (might have used in the past)
- past user: currently not using, but using in the past, not considering. 
- non-user: currently not using, not considering, not used in the past


In [ ]:
use_conditions = [
    ((data['current_use'] != 1) & (data["ever_use"] != 1)),  # Never used, not currently using
    (data['current_use'] == 1),  # Current user
    ((data['ever_use'] == 1) & (data['current_use'] != 1)),  # Past user: Ever used but not currently using
]

use_choices = ['nonuser', 'user', 'past_user']

In [ ]:
data['use'] = np.select(use_conditions, use_choices, default="NaN")

# Future users are a subcategory of nonusers so they need to be categorised after
# Not current, not ever, but considering
data.loc[((data['ever_use'] != 1) & (data['future_intent'] == 1)), 'use'] = 'future_user'

data.loc[data.use=='user'].groupby(['current_use', 'ever_use', 'future_intent']).count()

#### gender

In [ ]:
gender_cols = [col for col in data.columns if re.match(r'^gender_[0-9]+$', col)]
data['gender'] = data[gender_cols].bfill(axis=1).iloc[:, 0]
data=data.drop(columns=gender_cols)


#### age

In [ ]:
age_cols = [col for col in data.columns if re.match(r'^respondent_age_[0-9]+$', col)]
data['age'] = data[age_cols].bfill(axis=1).iloc[:, 0]
data=data.drop(columns=age_cols)

# bin age
bins = [15, 20, 30, 45]
labels = ['16-20', '21-30', '31-45']
data['age_group'] = pd.cut(data['age'], bins=bins, labels=labels, right=True)


#### travel time

In [ ]:
travel_col = data['willingness_to_travel'].astype(int)

bins = [0, 5,15, 30, 45, 60, 90, 120, 180, 240, 1500]
labels = [
    '0-5 minutes',
    '5-15 minutes',
    '15-30 minutes',
    '30-45 minutes',
    '45-60 minutes',
    '60-90 minutes',
    '90-120 minutes',
    '2-3 hours',
    '3-4 hours',
    '4+ hours'
    ]
data['willingness_to_travel'] = pd.cut(data['willingness_to_travel'], bins=bins, labels=labels, right=True)

random draw attempts

In [ ]:
# NOTE: was hardcoded as "16 - ...", but the raw export actually has 19
# random_draw_name_available_N columns (household_member_selection repeat can run up to
# 19 times), not 16. That silently shifted every respondent's attempt count down by 3
# (e.g. the common "first draw succeeded" case read as -2 instead of the correct 1).
random_draw_cols=[col for col in data.columns if re.match(r'^random_draw_name_available_[0-9]+$', col)]
data["random_draw_attempts"]=len(random_draw_cols)-data[random_draw_cols].isna().sum(axis=1)

#### Potential Cleanup

In [ ]:
# create merged column of index selected
# NOTE: also hardcoded to 16 previously -- same bug as random_draw_attempts above.
# Use the actual max repeat instance (19) so late-succeeding draws (attempt 17-19)
# aren't silently dropped from the bfill.
max_repeat = max(int(c.rsplit('_', 1)[1]) for c in data.columns if re.match(r'^random_index_[0-9]+$', c))
index_cols = [f'random_index_{i}' for i in range(max_repeat, 0, -1)]
data['index_selected'] = data[index_cols].bfill(axis=1).iloc[:, 0].astype(int)
data=data.drop(columns=index_cols)

In [ ]:
# drop household_names column
cols_to_drop=[col for col in data.columns if re.match(r'^household_names_[0-8]$', col)]
data=data.drop(columns=cols_to_drop)

# drop disaggreated member_index columns
cols_to_drop=[col for col in data.columns if re.match(r'^hh_member_index_[0-8]$', col)]
data=data.drop(columns=cols_to_drop)

# drop random draw name available
cols_to_drop=[col for col in data.columns if re.match(r'^random_draw_name_available_[0-8]$', col)]
data=data.drop(columns=cols_to_drop)

#drop not_selected
data=data.drop(columns="not_selected_1")

Drop additional columns where there were multiple eligble and/or interviewed respondents per HH.

In [ ]:
# preserve statement cols
statement_cols = [col for col in data.columns if col.startswith("statement_")]
statement_df = data[statement_cols]

# Drop expanded columns
drop_cols_per_resp = data.filter(regex=r'_[0-9]{1,2}$').columns
data = data.drop(columns=drop_cols_per_resp)

# add back statement columns
data = pd.concat([data, statement_df], axis=1)

In [ ]:
print(f"Dropped: {len(drop_cols_per_resp)} columns")

#### Check against manually reviewed for QA

In [ ]:
# Read manual review by Mohamed
qa = pd.read_csv(DIR_META/'Benin QA Quantitative survey - Full Data with QA.csv')

In [ ]:
# Does the filtered data match manual review

msng_from_qa = set(data['KEY'].unique()) - set(qa.loc[qa['Validation status'] == 'Validated']['Submission Key'].unique())
qa_msng = data.loc[data['KEY'].isin(msng_from_qa)]
qa_msng.to_csv(DIR_VALIDATE/"00_missing_from_qa.csv",index=False)

In [ ]:
# Does the manual review match filtered data

msng_from_data = set(qa.loc[qa['Validation status'] == 'Validated']['Submission Key'].unique()) - set(data['KEY'].unique())
data_msng = qa.loc[qa['Submission Key'].isin(msng_from_data)]
data_msng.to_csv(DIR_VALIDATE/"00_missing_from_data_dwnld.csv",index=False)

#### Map data

In [ ]:
#Load metadata and raw data
survey = pd.read_excel(DIR_META_DATA, sheet_name='survey')
choices = pd.read_excel(DIR_META_DATA, sheet_name='choices')

### Export Cleaned Data

In [ ]:
data.shape

In [ ]:
data.to_csv(DIR_CLEANED_DATA, index=False)